### Importy

In [6]:
import math
from scipy.stats import norm, t, chi2, binom
from typing import Literal, Optional, Any
from functools import cache

### Potrzebne funkcje

In [7]:
@cache
def choose(n: int, k: int) -> int:
    if k == 0 or k == n:
        return 1
    if k == 1 or k == n - 1:
        return n
    return choose(n - 1, k - 1) + choose(n - 1, k)

def no_estymator_wariancji(dane: list[float]) -> float:
    """
    Oblicza nieobciążony estymator wariancji dla podanej próby.

    Args:
        dane (list[float]): lista zawierająca wyniki pobrane z próby.

    Returns:
        float: Wartość nieobciążonego estymatora wariancji.

    Example:
        >>> NEW([1, 2, 3, 4, 5, 6, 7, 8, 9])
        7.5
    """
    n = len(dane)
    x_bar = sum(dane) / n
    sq_diff = list(map(lambda x: (x - x_bar)**2, dane))
    return sum(sq_diff) / (n - 1)

def hipoteza_sredniej_jedna_proba(
    probka: list[float],
    srednia_h0: float,
    poziom_istotnosci: float,
    odchylenie_populacji: Optional[float] = None,
    typ_hipotezy: Literal["L", "P", "O"] = "O"
) -> bool:
    n = len(probka)
    srednia_proby = sum(probka) / n

    if odchylenie_populacji is not None:
        sigma = odchylenie_populacji
        uzyj_rozkadu_normalnego = True
    else:
        sigma = math.sqrt(no_estymator_wariancji(probka)) if n > 1 else 0.0
        uzyj_rozkadu_normalnego = n > 30

    statystyka = (srednia_proby - srednia_h0) * math.sqrt(n) / sigma

    if uzyj_rozkadu_normalnego:
        if typ_hipotezy == 'L':
            war_kryt = norm.ppf(poziom_istotnosci)
            return bool(statystyka > war_kryt)
        elif typ_hipotezy == 'P':
            war_kryt = norm.ppf(1 - poziom_istotnosci)
            return bool(statystyka < war_kryt)
        else: # Obustronny
            war_kryt = norm.ppf(1 - poziom_istotnosci / 2)
            return bool(-war_kryt < statystyka < war_kryt)
    else:
        stopien_swobody = n - 1
        if typ_hipotezy == 'L':
            war_kryt = t.ppf(poziom_istotnosci, stopien_swobody)
            return bool(statystyka > war_kryt)
        elif typ_hipotezy == 'P':
            war_kryt = t.ppf(1 - poziom_istotnosci, stopien_swobody)
            return bool(statystyka < war_kryt)
        else: # Obustronny
            war_kryt = t.ppf(1 - poziom_istotnosci / 2, stopien_swobody)
            return bool(-war_kryt < statystyka < war_kryt)

## Zadanie 1

Niech $X$ będzie zmienną losową oznaczającą ilość wadliwych butelek w próbce $n$ butelek. Zmienna ta ma ***rozkład dwumianowy*** z prawdopodobieństwem sukcesu $p$ (sukces to wylosowanie wadliwej butelki). Wiemy, że

$$
    \mathbb{E}(X) = np\\
    \text{Var}(X) = np(1-p)
$$

wprowadźmy zmienną losową $\hat{p} = \frac{X}{n}$. Jest ona proporcją wadliwych butelek w próbie do wszystkich $n$ butelek. Mamy

$$
    \mathbb{E}(\hat{p}) = \mathbb{E}(\frac{X}{n}) = \frac{1}{n}\mathbb{E}(X) = p \\[5pt]
    \text{Var}(\hat{p}) = \text{Var}(\frac{X}{n}) = \frac{1}{n^2}\text{Var}(X) = \frac{p(1-p)}{n}
$$

Możemy więc policzyć statystykę
$$
    T = \frac{\hat{p} - p_{\text{hip}}}{\sqrt{\frac{p_{\text{hip}}(1-p_{\text{hip}})}{n}}}
$$
gdzie $p_{\text{hip}}$ oznacza proporcję z hipotezy

In [8]:
n = 900
wadliwe = 18
p_hip = 0.03
p_hat = wadliwe / n
print(f"{n = }")
print(f"{wadliwe = }")
print(f"{p_hip = }")
print(f"{p_hat = }")

T = (p_hat - p_hip) / math.sqrt(p_hip * (1 - p_hip) / n)
print(f"{T = }")

n = 900
wadliwe = 18
p_hip = 0.03
p_hat = 0.02
T = -1.7586311452816472


Ponieważ próba jest dość duża (900) korzystamy z rozkładu normalnego. Porównujemy naszą hipotezę zerową do hipotezy lewostronnej dla poziomu istotności $\alpha = 0.05$:

In [9]:
alfa = 0.05
wartosc_krytyczna = -float(norm.ppf(1 - alfa)) # '-' z powodu hipotezy lewostronnej
print(f"{wartosc_krytyczna = }")
print(f"{alfa = }")

wartosc_krytyczna = -1.6448536269514722
alfa = 0.05


Widzimy że $T < \textbf{wartość krytyczna}$ zatem odrzucamy hipotezę zerową na rzecz hipotezy alternatywnej (lewostronnej).

## Zadanie 2

Przyjmujemy: 
* Hipoteza zerowa - stosunek sztuk wadliwych do wszystkich $ \leq 0.06$
* Hipoteza alternatywna (prawostronna) - stosunek sztuk wadliwych do wszystkich $ > 0.06$

Zatem przypadek graniczny to $p=0.06$ \
Przyjmujemy poziom istotności $\alpha = 0.1$

Niech zmienna losowa $X$ oznacza ilość wadliwych sprzetów w $n=15$ wylosowanych sprzętach. Zmienna ta posiada ***rozkład dwumianowy*** z prawdopodobieństwem sukcesu $p=0.06$

Obliczamy p-wartość czyli najmniejszą wartość krytyczną, przy której możemy odrzucić hipotezę zerową. To znaczy prawdopodobieństwo tego, że wylosujemy coś tak złego lub gorszego od naszej próbki. Wynosi ona

$$
    \text{p-wartość} = P(X \ge 3) = 1 - P(X \le 2) = 1 - \sum_{k=0}^2 \binom{n}{k}p^k(1-p)^{n-k} 
$$


In [10]:
p_wartosc = 1
n = 15
p_hip = 0.06
alfa = 0.1

for k in range(3):
    p_wartosc -= choose(n, k) * p_hip**k * (1-p_hip) **(n - k)
print(f"{p_wartosc = }")
print(f"{alfa = }")


p_wartosc = 0.057133323827631505
alfa = 0.1


Widać że **p-wartość** $ < \alpha$ zatem odrzucamy hipotezę zerową na rzecz hipotezy alternatywnej 

## Zadanie 3

Niech $\hat{p}_A$ oznacza proporcję osób wyleczonych w grupie $A$ do wszystkich osób w grupie $A$, a $\hat{p}_B$ proporcję osób wyleczonych w grupie $B$ do wszystkich osób w grupie $B$. Rozmiary grup $n_A=n_B=n=20$. Zmienne $\hat{p}_A$ oraz $\hat{p}_B$ to zmienne o ***rozkładzie dwumianowym***\
Lek $B$ będzie skuteczniejszy, gdy procent osób wyleczonych za pomącą leku $B$, będzie większy od procent osób wyleczonych lekiem $A$.\
Przyjmujemy: 
* Hipoteza zerowa: $A$ i $B$ są równie skutecznie czyli $p_A = p_B = \bar{p} = \frac{11 + 17}{20 + 20} = 0.7$, gdzie $p_i$ - prawdopodobieństwo, że lek $i$ wyleczy pacjenta. Możemy policzyć łączną proporcję (estymator) $\bar{p}$, ponieważ przy powyższym założeniu grupy $A$ i $B$ niczym się nie różnią, czyli traktujemy je jako jedną dużą grupę.
* Hipoteza alternatywna: $B$ jest skuteczniejsze od $A$, czyli $p_B - p_A > 0$\
Możemy policzyć statystykę:

$$
    T = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\frac{2\bar{p}(1-\bar{p})}{n}}}
$$

Wzór ten wynika z rozważań podobnych do tych przytoczonych w zadaniu 1.



Przyjmujemy poziom istotności $\alpha = 0.01$

In [11]:
n = 20
alfa = 0.01
wyleczeni_A = 11
wyleczeni_B = 17
p_hatA = wyleczeni_A / n
p_hatB = wyleczeni_B / n
p_bar = (wyleczeni_A + wyleczeni_B) / (n + n)

T = (p_hatB - p_hatA) / math.sqrt((2 * p_bar * (1 - p_bar) / n))
print(f"{n = }")
print(f"{alfa = }")
print(f"{wyleczeni_A = }")
print(f"{wyleczeni_B = }")
print(f"{p_hatA = }")
print(f"{p_hatB = }")
print(f"{p_bar = }")
print(f"{T = }")

n = 20
alfa = 0.01
wyleczeni_A = 11
wyleczeni_B = 17
p_hatA = 0.55
p_hatB = 0.85
p_bar = 0.7
T = 2.070196678027062


W tej sytuacji bierzmy wartości krytyczne z ***rozkładu normalnego***

In [12]:
wartosc_krytyczna = float(norm.ppf(1 - alfa)) # hipoteza prawostronna
print(f"{wartosc_krytyczna = }")


wartosc_krytyczna = 2.3263478740408408


Widzimy, że $T<\text{wartość krytyczna}$ co nie daje nam podstaw do odrzucenia hipotezy zerowej. Nie możemy zatem powiedzieć, że lek $B$ jest skuteczniejszy od leku $A$